# Business Applications Gallery: Bandits Across Domains

**Goal:** Explore diverse business applications of bandits beyond content and conversion optimization. Learn to frame any sequential decision problem as a bandit.

**What you'll learn:**
- Pricing optimization with revenue-aware bandits
- Email send-time optimization for open rates
- Trading alert threshold optimization for commodity traders
- A template for designing your own business bandit

**Time:** 15 minutes

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from collections import defaultdict

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
print("Ready to explore business bandits!")

In [ ]:
apply_course_theme()
apply_plot_theme()

## Application 1: Pricing Optimization

**Problem:** You're selling a commodity research subscription. Which price point maximizes revenue?

**Arms:** 3 price points ($39, $59, $79 per month)

**Reward:** Revenue per visitor (price × conversion rate)

**Key insight:** Higher price ≠ higher revenue. Demand curve matters.

In [ ]:
# Pricing bandit setup
prices = [39, 59, 79]

# Demand curve: higher price → lower conversion
conversion_rates = {
    39: 0.08,  # Low price, high conversion
    59: 0.06,  # Mid price, mid conversion
    79: 0.03   # High price, low conversion
}

# Expected revenue per visitor
expected_revenue = {
    price: price * conversion_rates[price] 
    for price in prices
}

print("Pricing Strategy Analysis:")
print("=" * 50)
for price in prices:
    conv = conversion_rates[price]
    rev = expected_revenue[price]
    print(f"${price}/mo: {conv:.1%} conversion → "
          f"${rev:.2f} per visitor")

best_price = max(expected_revenue, key=expected_revenue.get)
print(f"\nOptimal price: ${best_price} "
      f"(${expected_revenue[best_price]:.2f}/visitor)")

In [ ]:
# Thompson Sampling for pricing
class PricingBandit:
    def __init__(self, prices):
        self.prices = prices
        self.total_revenue = {p: 0 for p in prices}
        self.n_shown = {p: 0 for p in prices}
        self.revenue_squared = {p: 0 for p in prices}
        
    def select_price(self):
        # Gaussian Thompson Sampling
        samples = {}
        for price in self.prices:
            if self.n_shown[price] < 2:
                # Optimistic init for new prices
                samples[price] = 10.0
            else:
                mean = (self.total_revenue[price] / 
                       self.n_shown[price])
                # Estimate variance
                var = max(0.1, 
                         (self.revenue_squared[price] / 
                          self.n_shown[price] - mean**2))
                std = np.sqrt(var / self.n_shown[price])
                samples[price] = np.random.normal(mean, std)
        return max(samples, key=samples.get)
    
    def update(self, price, revenue):
        self.n_shown[price] += 1
        self.total_revenue[price] += revenue
        self.revenue_squared[price] += revenue**2

# Simulate 5000 visitors
bandit = PricingBandit(prices)
n_visitors = 5000

for visitor in range(n_visitors):
    price = bandit.select_price()
    # Simulate conversion
    converted = (np.random.rand() < 
                conversion_rates[price])
    revenue = price if converted else 0
    bandit.update(price, revenue)

print("\nPRICING BANDIT RESULTS (5000 visitors):")
print("=" * 50)
total_revenue = 0
for price in prices:
    shown = bandit.n_shown[price]
    rev = bandit.total_revenue[price]
    avg = rev / shown if shown > 0 else 0
    print(f"${price}: {shown} shown, ${rev:.0f} revenue "
          f"(${avg:.2f}/visitor)")
    total_revenue += rev

print("=" * 50)
print(f"Total revenue: ${total_revenue:.0f}")

# Compare to random pricing
random_revenue = n_visitors * np.mean(
    list(expected_revenue.values())
)
print(f"Random pricing: ${random_revenue:.0f}")
print(f"Lift: ${total_revenue - random_revenue:.0f} "
      f"({(total_revenue/random_revenue - 1):.1%})")

## Application 2: Email Send-Time Optimization

**Problem:** When should you send your weekly commodity market newsletter for maximum open rates?

**Arms:** 7 send times (Mon-Sun, 8am EST)

**Reward:** Open rate

**Context:** Traders are busy during market hours, free on weekends

In [ ]:
# Email timing optimization
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

# True open rates by day
open_rates = {
    'Mon': 0.22,  # Start of week, low
    'Tue': 0.28,  # Mid-week, better
    'Wed': 0.32,  # Peak workday attention
    'Thu': 0.29,
    'Fri': 0.18,  # Pre-weekend distraction
    'Sat': 0.15,  # Weekend, very low
    'Sun': 0.25   # Sunday prep for week
}

class EmailTimingBandit:
    def __init__(self, days):
        self.days = days
        self.alpha = {d: 1 for d in days}  # Beta prior
        self.beta = {d: 1 for d in days}
        
    def select_day(self):
        samples = {
            d: np.random.beta(self.alpha[d], self.beta[d]) 
            for d in self.days
        }
        return max(samples, key=samples.get)
    
    def update(self, day, opened):
        if opened:
            self.alpha[day] += 1
        else:
            self.beta[day] += 1

# Simulate 52 weeks (1 year)
email_bandit = EmailTimingBandit(days)
subscribers = 1000
week_history = []

for week in range(52):
    send_day = email_bandit.select_day()
    true_rate = open_rates[send_day]
    
    # Simulate opens
    opens = sum(np.random.rand() < true_rate 
               for _ in range(subscribers))
    
    # Update bandit (batch update)
    email_bandit.alpha[send_day] += opens
    email_bandit.beta[send_day] += (subscribers - opens)
    
    week_history.append({
        'week': week,
        'day': send_day,
        'opens': opens,
        'rate': opens / subscribers
    })

# Analyze results
df = pd.DataFrame(week_history)
print("EMAIL SEND-TIME OPTIMIZATION (52 weeks):")
print("=" * 50)
print("\nWeeks sent by day:")
print(df['day'].value_counts().sort_index())

print("\nLearned open rates (posterior means):")
for day in days:
    a = email_bandit.alpha[day]
    b = email_bandit.beta[day]
    learned = a / (a + b)
    true = open_rates[day]
    print(f"{day}: {learned:.1%} (true: {true:.1%})")

# Calculate lift
total_opens = df['opens'].sum()
avg_rate = np.mean(list(open_rates.values()))
random_opens = 52 * subscribers * avg_rate
print(f"\nTotal opens (bandit): {total_opens:.0f}")
print(f"Random day opens: {random_opens:.0f}")
print(f"Lift: {total_opens - random_opens:.0f} opens")

In [ ]:
# Visualize day selection over time
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Plot 1: Day selection timeline
day_colors = {d: plt.cm.tab10(i) for i, d in enumerate(days)}
for i, row in df.iterrows():
    ax1.scatter(row['week'], row['rate'], 
               color=day_colors[row['day']], 
               s=100, alpha=0.6)

# Add legend
for day in days:
    ax1.scatter([], [], color=day_colors[day], 
               label=f"{day} ({open_rates[day]:.0%})")

ax1.set_xlabel('Week')
ax1.set_ylabel('Observed Open Rate')
ax1.set_title('Email Send Day Selection Over 52 Weeks')
ax1.legend(ncol=7, loc='upper right', fontsize=8)
ax1.grid(True, alpha=0.3)

# Plot 2: Cumulative sends by day
day_counts = df['day'].value_counts().sort_index()
ax2.bar(day_counts.index, day_counts.values, 
       color=[day_colors[d] for d in day_counts.index])
ax2.set_xlabel('Day of Week')
ax2.set_ylabel('Weeks Sent')
ax2.set_title('Final Allocation: Bandit Favors Wed')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Application 3: Trading Alert Threshold Optimization

**Problem:** You send crude oil volatility alerts. Which threshold generates the most actionable trades?

**Arms:** 4 volatility thresholds (1.5σ, 2σ, 2.5σ, 3σ)

**Reward:** % of alerts that lead to profitable trades

**Trade-off:** Low threshold = many alerts (noisy), high threshold = few alerts (miss moves)

In [ ]:
# Trading alert optimization
thresholds = ['1.5σ', '2.0σ', '2.5σ', '3.0σ']

# Alert quality metrics
alert_stats = {
    '1.5σ': {'frequency': 0.20, 'actionable': 0.30},  # Many, low quality
    '2.0σ': {'frequency': 0.12, 'actionable': 0.55},  # Balanced
    '2.5σ': {'frequency': 0.06, 'actionable': 0.70},  # Best quality
    '3.0σ': {'frequency': 0.02, 'actionable': 0.75}   # Rare, very good
}

class AlertBandit:
    def __init__(self, thresholds):
        self.thresholds = thresholds
        self.alpha = {t: 1 for t in thresholds}
        self.beta = {t: 1 for t in thresholds}
        self.alerts_sent = {t: 0 for t in thresholds}
        self.actionable_count = {t: 0 for t in thresholds}
        
    def select_threshold(self):
        samples = {
            t: np.random.beta(self.alpha[t], self.beta[t]) 
            for t in self.thresholds
        }
        return max(samples, key=samples.get)
    
    def simulate_day(self, threshold):
        """Simulate a day with this threshold"""
        stats = alert_stats[threshold]
        # Does an alert trigger today?
        if np.random.rand() < stats['frequency']:
            self.alerts_sent[threshold] += 1
            # Is it actionable?
            actionable = (np.random.rand() < 
                         stats['actionable'])
            if actionable:
                self.actionable_count[threshold] += 1
                self.alpha[threshold] += 1
            else:
                self.beta[threshold] += 1
            return 1, actionable
        return 0, False

# Simulate 250 trading days (1 year)
alert_bandit = AlertBandit(thresholds)
threshold_usage = defaultdict(int)

for day in range(250):
    threshold = alert_bandit.select_threshold()
    threshold_usage[threshold] += 1
    alert_bandit.simulate_day(threshold)

print("TRADING ALERT THRESHOLD OPTIMIZATION (250 days):")
print("=" * 60)
print("\nThreshold usage (days active):")
for t in thresholds:
    print(f"  {t}: {threshold_usage[t]} days")

print("\nAlert performance:")
for t in thresholds:
    sent = alert_bandit.alerts_sent[t]
    actionable = alert_bandit.actionable_count[t]
    if sent > 0:
        rate = actionable / sent
        true_rate = alert_stats[t]['actionable']
        print(f"  {t}: {sent} alerts, {actionable} actionable "
              f"({rate:.1%}, true: {true_rate:.1%})")
    else:
        print(f"  {t}: No alerts sent")

print("\nPosterior beliefs (actionable rate):")
for t in thresholds:
    a = alert_bandit.alpha[t]
    b = alert_bandit.beta[t]
    belief = a / (a + b)
    print(f"  {t}: {belief:.1%}")

## Design Your Own Business Bandit

Use this template to frame any business problem as a bandit.

In [ ]:
# TEMPLATE: Business Bandit Design

print("BUSINESS BANDIT DESIGN TEMPLATE")
print("=" * 60)

template = """
1. PROBLEM STATEMENT
   What decision do you make repeatedly?
   Example: "Which blog topic should I write about this week?"

2. ARMS (OPTIONS)
   What are your choices? (Must be repeatable)
   Example: ["Market Analysis", "Trading Psychology", "Tools"]
   
   ✓ Repeatable weekly/daily
   ✓ Mutually exclusive (pick one per decision)
   ✓ Finite set (≤ 10 arms ideal)

3. REWARD METRIC
   What defines success for this decision?
   Example: "Read ratio (% who finish article)"
   
   ✓ Observable quickly (hours/days, not months)
   ✓ Aligned with business goal (not vanity metric)
   ✓ Numeric (can compare arms)

4. CONTEXT (Optional)
   What factors might affect the best choice?
   Example: "Day of week, market volatility, subscriber segment"
   
   → If context matters, use Contextual Bandits (Module 3)

5. CONSTRAINTS
   Any hard limits or safety requirements?
   Example: "Must publish at least 1 Tools post per month"
   
   → Add constraint to arm selection logic

6. ALGORITHM CHOICE
   Binary reward? → Thompson Sampling (Beta-Bernoulli)
   Continuous reward? → Thompson Sampling (Gaussian)
   Need interpretability? → UCB1
   Non-stationary? → Discounted Thompson Sampling

7. EXPLORATION BUDGET
   How much can you afford to experiment?
   Example: "20% of posts to non-top arms"
   
   High stakes → 10% exploration
   Low stakes → 30% exploration

8. EVALUATION PLAN
   How will you measure if the bandit is working?
   Example: "Compare avg read ratio to pre-bandit baseline"
   
   ✓ Baseline metric (before bandits)
   ✓ Target improvement (e.g., +15%)
   ✓ Evaluation period (e.g., 3 months)
"""

print(template)

## Summary Table: All Applications

Quick reference for bandit use cases across domains.

In [ ]:
# Summary of all applications
applications = pd.DataFrame([
    {
        'Use Case': 'Content Strategy',
        'Arms': 'Topic × Format',
        'Reward': 'Read ratio',
        'Algorithm': 'Thompson (Gaussian)',
        'Typical Lift': '15-25%'
    },
    {
        'Use Case': 'Landing Page',
        'Arms': 'Page variants',
        'Reward': 'Conversion',
        'Algorithm': 'Thompson (Beta)',
        'Typical Lift': '10-20%'
    },
    {
        'Use Case': 'Pricing',
        'Arms': 'Price points',
        'Reward': 'Revenue/visitor',
        'Algorithm': 'Thompson (Gaussian)',
        'Typical Lift': '5-15%'
    },
    {
        'Use Case': 'Email Timing',
        'Arms': 'Send times',
        'Reward': 'Open rate',
        'Algorithm': 'Thompson (Beta)',
        'Typical Lift': '8-12%'
    },
    {
        'Use Case': 'Alert Thresholds',
        'Arms': 'Trigger levels',
        'Reward': 'Actionable %',
        'Algorithm': 'Thompson (Beta)',
        'Typical Lift': '20-30%'
    },
    {
        'Use Case': 'Onboarding Flow',
        'Arms': 'Step variants',
        'Reward': 'Completion',
        'Algorithm': 'Thompson (Beta)',
        'Typical Lift': '10-18%'
    },
    {
        'Use Case': 'Ad Creative',
        'Arms': 'Creative variants',
        'Reward': 'CTR × Conv',
        'Algorithm': 'Thompson (Gaussian)',
        'Typical Lift': '12-22%'
    }
])

print("\nBUSINESS BANDIT APPLICATIONS SUMMARY")
print("=" * 80)
print(applications.to_string(index=False))
print("=" * 80)
print("\nNote: 'Lift' = improvement vs random/A-B test allocation")

## Commodity Trading Specific: Report Format Bandit

Bonus example for commodity traders.

In [ ]:
# Commodity report format optimization
formats = ['PDF Report', 'Email Digest', 'Video', 'Dashboard']

# Trader engagement metrics
format_performance = {
    'PDF Report': {
        'open_rate': 0.45,
        'action_rate': 0.12  # % who trade
    },
    'Email Digest': {
        'open_rate': 0.68,
        'action_rate': 0.22  # Best!
    },
    'Video': {
        'open_rate': 0.52,
        'action_rate': 0.15
    },
    'Dashboard': {
        'open_rate': 0.35,
        'action_rate': 0.18
    }
}

# Composite reward: open × action (both matter)
composite_scores = {
    fmt: stats['open_rate'] * stats['action_rate']
    for fmt, stats in format_performance.items()
}

print("COMMODITY REPORT FORMAT OPTIMIZATION")
print("=" * 60)
print("\nFormat performance:")
for fmt in formats:
    stats = format_performance[fmt]
    composite = composite_scores[fmt]
    print(f"\n{fmt}:")
    print(f"  Open rate: {stats['open_rate']:.1%}")
    print(f"  Action rate: {stats['action_rate']:.1%}")
    print(f"  Composite score: {composite:.3f}")

best = max(composite_scores, key=composite_scores.get)
print(f"\n→ Best format: {best} "
      f"(composite: {composite_scores[best]:.3f})")
print("\nKey insight: Email Digest has highest engagement × action")
print("even though PDF has higher perceived 'quality'")

## Summary

**Key Takeaways:**

1. **Bandits work across domains** — content, conversion, pricing, timing, alerts
2. **The framework is always the same** — define arms, choose reward, run Thompson Sampling
3. **Biggest wins:** Clear performance gaps, high opportunity cost, repeatable decisions
4. **For commodity traders:** Optimize alert thresholds, report formats, client communication

**Design checklist:**
- ✅ Repeatable arms (not one-time decisions)
- ✅ Quality reward metric (aligned with business goal)
- ✅ Fast feedback (hours/days, not months)
- ✅ Exploration budget (10-30%)
- ✅ Baseline for comparison (measure lift)

**When NOT to use bandits:**
- One-time decisions (use decision analysis)
- Too many arms (K > 20, try successive elimination)
- Delayed feedback (> 1 month, use proxy metrics)
- Need statistical significance (use A/B test)

**Next steps:**
- Pick one business problem from your work
- Fill out the design template above
- Start with 3-5 arms, simple reward metric
- Run for 4-8 weeks, measure lift vs baseline

**Module 5 Preview:** Apply these same principles to commodity portfolio allocation — the ultimate high-stakes bandit problem.

In [ ]:
key_takeaways(["Bandits work across domains", "The framework is always the same", "Biggest wins:", "For commodity traders:"])